In [ ]:
from google.cloud import storage, bigquery
import pandas as pd
from pyspark.sql import SparkSession
import datetime
import json

# Initializing Spark session
spark = SparkSession.builder.appName("RetailerMysqltoLanding").getOrCreate()

# Google Cloud Storage (GCS) configuration
gcs_bucket = "retailer-datalake-project-270326"
landing_file_path = f"gs://{gcs_bucket}/landing/retailer-db"
archive_file_path = f"gs://{gcs_bucket}/landing/retailer-db/archive/"
config_file_path = f"gs://{gcs_bucket}/configs/retailer_config.csv"

# BigQuery (BQ) configuration
bq_project = "gcp-demo"
bq_audit_table = f"{bq_project}.temp_dataset.audit_log"
bq_log_table = f"{bq_project}.temp_dataset.pipeline_logs"
bq_temp_path = f"{gcs_bucket}/temp/"

# MySQL Configuration
mysql_config = {
    "url": "jdbc:mysql://34.173.109.35:3306/retailer-db?useSSL=true&allowPublicKeyRetrieval=true",
    #"url": "jdbc:mysql://34.132.173.21:3306/retailerDB?useSSL=false&allowPublicKeyRetrieval=true",
    "driver": "com.mysql.cj.jdbc.Driver",
#     "driver": "com.mysql.cj.jdbc.Driver",
    "user": "myusr",
    "password": "Mypass@2026"
}

# Initializing GCS and BigQuery clients
storage_client = storage.Client()
bigquery_client = bigquery.Client()

# Logging mechanism
log_entries = []

def log_event(event_type, message, table=None):
    """Log event and store it in log list"""
    log_entry = {
        "time_stamp": datetime.datetime.now().isoformat(),
        "event_type": event_type,
        "message": message,
        "table": table
    }
    log_entries.append(log_entry)
    print(f"[{log_entry['time_stamp']}] {event_type} - {message}")

# Function to read the config file
def read_config_file():
    df = spark.read.csv(config_file_path, header=True)
    log_event("Info", f"✅Successfully read the config file")
    return df

#move file to archive
def move_existing_file_to_archive(table):
    blobs = list(storage_client.bucket(gcs_bucket).list_blobs(prefix=f"landing/retailer-db/{table}"))
    existing_file = [blob.name for blob in blobs if blob.name.endswith(".json") ]
    if not  existing_file:
        log_event("Info",f"No Existing file for table {table}")
    for file in existing_file:
        source_blob =  storage_client.bucket(gcs_bucket).blob(file)
        #extract date from the file name from table_
        date_part = file.split("_")[-1].split(".")[0]
        year,month,day = date_part[-4:],date_part[2:4],date_part[:2]
        #creating archive path & move to archive path
        archive_file_path = f"landing/retailer-db/archive/{table}/{year}/{month}/{day}/{file.split('/')[-1]}"
        destination_blob = storage_client.bucket(gcs_bucket).blob(archive_file_path)
        #copy file to archive and delete in originak file
        storage_client.bucket(gcs_buceket).copy_blob(source_blob,storage_client.bucket(gcs_bucket),destination_blob.name)
        source_blob.delete()
        log_event("Info",f"move {file} to {archive_file_path}",table= table)

#function to get latest watermark
def get_latest_watermark(table_name):
        query = f"""
                  select MAX(load_timestamp) as latest_timestamp
                  from `{bq_audit_table}`
                  where table_name = '{table_name}'
                 """
        query_job = bigquery_client.query(query)
        result = query_job.result()
        for row in result:
             return row.latest_timestamp if row.latest_timestamp else "1900-01-01 00:00:00"
        return "1900-01-01 00:00:00"
#Function to extract data from mysql to gcp
def extract_and_save_to_landing(table, load_type, watermark_col):
    try:
        # get latest watermark from the audit table
        last_watermark = get_latest_watermark(table) if load_type.lower() == 'incremental' else None
        log_event("Info", f"Latest watermark for {table}: {last_watermark}", table=table)

        # generate SQL query
        query = f"(SELECT * FROM {table}) AS t" if load_type.lower() == "full load" else \
                f"(SELECT * FROM {table} WHERE {watermark_col} > '{last_watermark}') AS t"

        # read data from MySQL
        df = (spark.read
                .format("jdbc")
                .option("url", mysql_config["url"])
                .option("user", mysql_config["user"])
                .option("password", mysql_config["password"])
                .option("driver", mysql_config["driver"])
                .option("dbtable", query)
                .load())
        log_event("SUCCESS", f"✅ Successfully extracted data from {table}", table=table)

        # convert Spark dataframe into pandas dataframe then to JSON
        pandas_df = df.toPandas()
        json_data = pandas_df.to_json(orient="records", lines=True)

        # generate file path to GCS
        today = datetime.datetime.today().strftime('%d%m%Y')
        json_file_path = f"landing/retailer-db/{table}/{table}_{today}.json"

        # upload JSON to GCS
        bucket = storage_client.bucket(gcs_bucket)
        blob = bucket.blob(json_file_path)
        blob.upload_from_string(json_data, content_type="application/json")

        log_event("SUCCESS", f"✅ JSON file successfully written to gs://{gcs_bucket}/{json_file_path}", table=table)

        # insert audit entry
        audit_df = spark.createDataFrame([
            (table, load_type, df.count(), datetime.datetime.now(), "SUCCESS")
        ], ["tablename", "load_type", "record_count", "load_timestamp", "status"])

        (audit_df.write.format("bigquery")
             .option("table", bq_audit_table)
             .option("temporaryGcsBucket", gcs_bucket)
             .mode("append")
             .save())
        log_event("SUCCESS", f"✅ Audit log uploaded for {table}", table=table)

    except Exception as e:
        log_event("ERROR", f"Error processing {table}: {str(e)}", table=table)

# Main execution
config_df = read_config_file()
for row in config_df.collect():
     if row['is_active'] =='1':
            db,scr,table,load_type,watermark,_,targetpath = row
            move_existing_file_to_archive(table)
            extract_and_save_to_landing(table,load_type,watermark)